# Unit 9: Foundation Models: Data Preparation

This unit builds a small **video-understanding pipeline** on a German
documentary clip from the [Planet Wissen](https://commons.wikimedia.org/wiki/Category:Planet_Wissen)
series on Wikimedia Commons, using three open models:

| Stage | Model | Notebook |
|-------|-------|----------|
| Speech → text | [`openai/whisper-large-v3`](https://huggingface.co/openai/whisper-large-v3) | this one (`00`) |
| German → English translation | [`Qwen/Qwen2.5-VL-7B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct) | `01_Translate` |
| Summary + on-screen-text OCR | [`Qwen/Qwen2.5-VL-7B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct) | `02_Summarize_OCR` |

In this first notebook we **download** one clip, **convert** it to MP4, and
**transcribe** the German speech with Whisper Large-V3.

**Sources**
- Video: [Der_Golfstrom_-_Planet_Wissen.webm](https://commons.wikimedia.org/wiki/File:Der_Golfstrom_-_Planet_Wissen.webm) — Planet Wissen, Wikimedia Commons
- Whisper Large-V3: <https://huggingface.co/openai/whisper-large-v3> ([paper](https://arxiv.org/abs/2212.04356))
- Qwen2.5-VL: <https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct> ([report](https://arxiv.org/abs/2502.13923))
- FFmpeg: <https://ffmpeg.org/> · `qwen-vl-utils`: <https://pypi.org/project/qwen-vl-utils/>

## Prerequisites

**1. FFmpeg** (used to convert the video and to extract audio/frames). It is a
system package, *not* a Python one, so install it with `apt`:

```bash
sudo apt update
sudo apt install -y ffmpeg
ffmpeg -version      # verify it is on your PATH
```

**2. The Python environment** for this unit (managed with [uv](https://docs.astral.sh/uv/)).
From this directory:

```bash
uv sync                 # creates .venv with torch (CUDA 12.8), transformers, etc.
```

`**GPU:** Whisper Large-V3 (~3 GB) and Qwen2.5-VL-7B (~16 GB in fp16) are large.`

`The models are loaded one at a time and`

`unloaded afterwards so they never share GPU memory.`

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

# Make the vendored `video_pipeline` package importable even if the project
# was not installed (e.g. running `jupyter` outside `uv run`).
SRC = Path.cwd() / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PREPARED = Path("prepared")
PREPARED.mkdir(exist_ok=True)

if shutil.which("ffmpeg") is None:
    raise SystemExit(
        "ffmpeg not found on PATH. Install it first (see the cell above):\n"
        "    sudo apt update && sudo apt install -y ffmpeg"
    )
print("ffmpeg:", shutil.which("ffmpeg"))

## 1. Download the video

We pull the `.webm` straight from Wikimedia Commons with `wget`. The
`Special:FilePath` URL redirects to the current media file, so we don't have to
hard-code the storage path.

In [ ]:
COMMONS_TITLE = "Der_Golfstrom_-_Planet_Wissen.webm"
STEM = "der_golfstrom"

webm_path = PREPARED / f"{STEM}.webm"
url = f"https://commons.wikimedia.org/wiki/Special:FilePath/{COMMONS_TITLE}"
if not webm_path.exists():
    subprocess.run(["wget", "-nv", "-c", "-O", str(webm_path), url], check=True)
else:
    print("already downloaded")
print(webm_path, webm_path.stat().st_size // 1024, "KiB")

## 2. Convert WebM → MP4

The pipeline works with `.mp4`/`.mkv` containers. We re-encode the VP8/Vorbis
WebM to **H.264 video + AAC audio**, which is the most widely compatible MP4
profile. This is the FFmpeg command from the project brief:

```bash
ffmpeg -i input.webm -c:v libx264 -c:a aac -strict -2 output.mp4
```

In [ ]:
mp4_path = PREPARED / f"{STEM}.mp4"
if not mp4_path.exists():
    cmd = [
        "ffmpeg", "-y", "-loglevel", "error", "-nostdin", "-i", str(webm_path),
        "-c:v", "libx264", "-c:a", "aac", "-strict", "-2",
        str(mp4_path),
    ]
    subprocess.run(cmd, check=True)
else:
    print("already converted")
print(mp4_path, mp4_path.stat().st_size // 1024, "KiB")

In [ ]:
from video_pipeline.audio_extractor import probe_duration

duration = probe_duration(mp4_path)
print(f"duration: {duration:.1f} s")

## 3. Transcribe the German speech (Whisper Large-V3)

`WhisperPipeline` extracts a 16 kHz mono WAV with FFmpeg and runs Whisper
Large-V3, returning timestamped segments. We force `language="german"` so the
detector can't drift. The model is unloaded from the GPU on `with`-block exit.

> Equivalent one-liner on the command line:
> ```bash
> uv run video-pipeline whisper prepared/STEM.mp4 -l german -o prepared/transcript.json
> ```

In [ ]:
from video_pipeline.whisper_pipeline import WhisperPipeline

transcript_path = PREPARED / "transcript.json"
with WhisperPipeline() as wp:
    transcript = wp.transcribe(mp4_path, language="german")

transcript_path.write_text(transcript.model_dump_json(indent=2), encoding="utf-8")
print(f"{len(transcript.segments)} segments, {len(transcript.text)} chars -> {transcript_path}")

In [ ]:
# Peek at the first few timestamped segments.
for seg in transcript.segments[:6]:
    print(f"[{seg.start:6.1f} - {seg.end:6.1f}s]  {seg.text}")

---
**Next:** `01_Translate.ipynb` — translate this German transcript into English
with Qwen2.5-VL.